# Belle II Continuum Suppression

This notebook is the project's sequential learning artifact. It explains the physics and orchestrates tested functions from `src/` in this order: data ingestion → audit → cleaning → EDA → splitting → baselines → six classifiers → feature reduction → evaluation.

The raw dataset is private and is never committed. Until it is placed in `data/raw/`, data-dependent cells explain what is waiting instead of failing.

## 1. Imports and reproducibility

Reusable behavior belongs in `src/`; this notebook imports and demonstrates it. A single random seed and test fraction will be reused by every model for a fair comparison.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, RANDOM_STATE, RAW_DATA_DIR, TEST_SIZE
from src.data_io import assign_split, audit_frame, load_raw_to_db
from src.db import read_table, table_exists

print(f"Project root: {PROJECT_ROOT}")
print(f"Random state: {RANDOM_STATE}; test fraction: {TEST_SIZE}")

## 2. Locate the private dataset

The untouched CSV belongs in `data/raw/`. Exactly one CSV is selected automatically; if none or several are present, the notebook stops the data-dependent sequence safely and explains what to fix.

In [ ]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))
raw_csv_path = csv_files[0] if len(csv_files) == 1 else None

if not csv_files:
    print("Waiting for the private Belle II CSV in data/raw/.")
elif len(csv_files) > 1:
    print("Multiple CSV files found; select the intended source explicitly before ingestion:")
    for path in csv_files:
        print(f"  - {path.name}")
else:
    print(f"Dataset found: {raw_csv_path.name}")

## 3. Immutable CSV → SQLite ingestion

`raw_events` is the database copy of the source and must not be silently replaced. On later runs, we read the existing table instead of rewriting it. This demonstrates database writing and reading while preserving raw-data provenance.

In [ ]:
raw_events = None
if raw_csv_path is not None:
    if table_exists("raw_events", DB_PATH):
        raw_events = read_table("raw_events", DB_PATH)
        print("Read the existing immutable raw_events table.")
    else:
        raw_events = load_raw_to_db(raw_csv_path, DB_PATH)
        print("Wrote the immutable raw_events table.")
    print(f"Loaded {len(raw_events):,} rows and {raw_events.shape[1]} columns.")
else:
    print("Ingestion skipped until exactly one source CSV is available.")

## 4. Raw-data audit before cleaning

We inspect shape, types, missing and non-finite values, duplicates, and constant columns without changing any event. These facts will drive explicit cleaning decisions after the real schema is known.

In [ ]:
raw_audit = audit_frame(raw_events) if raw_events is not None else None
if raw_audit is not None:
    print(f"Rows: {raw_audit['rows']:,}; columns: {raw_audit['columns']}")
    print(f"Duplicate rows: {raw_audit['duplicate_rows']:,}")
    print(f"Constant columns: {raw_audit['constant_columns']}")
    display(raw_events.head())
else:
    print("Audit skipped because no dataset has been loaded.")

## 5. Persist one fair train/test split

The label name and encoding must be confirmed from the actual dataset—never guessed. After confirmation, `assign_split` creates one stratified assignment that every classifier reuses. Learned imputation, scaling, PCA, and feature selection will fit on training rows only.

In [ ]:
LABEL_COLUMN = None  # Set only after the real schema and physics meaning are confirmed.
events_with_split = None
if raw_events is not None and LABEL_COLUMN is not None:
    events_with_split = assign_split(raw_events, LABEL_COLUMN)
    display(events_with_split.groupby(["split", LABEL_COLUMN]).size().rename("events"))
else:
    print("Split assignment waits for dataset inspection and label confirmation.")

## Next: cleaning and physics-led EDA

Once the private data arrives, the next contribution will document the real label, event identifier, optional MC weight, and Belle II feature mapping. Cleaning and EDA sections will then be implemented here and in `src/` together before any model is trained.